In [72]:
from TinyViT.models.tiny_vit import tiny_vit_5m_224
model = tiny_vit_5m_224(pretrained=True)

/home/a01328525/transformer_object_dectection/TinyViT/models/tiny_vit.py:645: UserWarning: Overwriting tiny_vit_5m_224 in registry with TinyViT.models.tiny_vit.tiny_vit_5m_224. This is because the name being registered conflicts with an existing name. Please check if this is not expected.
  def tiny_vit_5m_224(pretrained=False, **kwargs):
/home/a01328525/transformer_object_dectection/TinyViT/models/tiny_vit.py:658: UserWarning: Overwriting tiny_vit_11m_224 in registry with TinyViT.models.tiny_vit.tiny_vit_11m_224. This is because the name being registered conflicts with an existing name. Please check if this is not expected.
  def tiny_vit_11m_224(pretrained=False, **kwargs):
/home/a01328525/transformer_object_dectection/TinyViT/models/tiny_vit.py:671: UserWarning: Overwriting tiny_vit_21m_224 in registry with TinyViT.models.tiny_vit.tiny_vit_21m_224. This is because the name being registered conflicts with an existing name. Please check if this is not expected.
  def tiny_vit_21m_224(

In [3]:
model

TinyViT(
  (patch_embed): PatchEmbed(
    (seq): Sequential(
      (0): Conv2d_BN(
        (c): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): GELU(approximate='none')
      (2): Conv2d_BN(
        (c): Conv2d(32, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
    )
  )
  (layers): ModuleList(
    (0): ConvLayer(
      (blocks): ModuleList(
        (0-1): 2 x MBConv(
          (conv1): Conv2d_BN(
            (c): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
            (bn): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          )
          (act1): GELU(approximate='none')
          (conv2): Conv2d_BN(
            (c): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), pa

In [73]:
import torch
import model.trainer as trainer
import utils.utils
#root = 'E:/Experiments/'
root = '/home/a01328525/'

folder_maps = root+'Datasets Maps/'
folder_save_results = root+'Counting images paper/'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

channel_selection = 'bgr'#'bgr_vis'#'bgr'
model_dir = root+"Datasets_STED/runs_transformers/STED_101_nano_cbbox_bgr_diou_BitNet/best.pt"
N_channels = 3
n_model = 128
num_blks = 1
loss_type = 'diou'
obj = 'cbbox'
conf_thr = 0.4; iou_thr=0.5; diou_thr=0.5

batch_size=16 
augment = False
probAugment = 0

train_dataloader, test_dataloader = trainer.get_dataloaders(root+'/Datasets_STED/Zones_cbbox_dataset_full_10/', obj,
                                                    augment, probAugment, batch_size)

Load train data with 3026 images
Load val data with 1377 images


In [74]:
for i in test_dataloader:
    P, imgs, targets, masks =  i
    break
print(imgs.shape)

torch.Size([16, 3, 224, 224])


In [75]:
#output = model(imgs)
features = model.forward_features(imgs)  # shape: (3, 197, 192) for TinyViT-5M
#cls_token = features[:, 0, :]  # Global feature: (3, 192)
#features.shape

In [78]:
import numpy as np
model_parameters = filter(lambda p: p.requires_grad, model.parameters())
params = sum([np.prod(p.size()) for p in model_parameters])
params

5392764

In [76]:
for feat in features:
    print(feat.shape)

torch.Size([16, 784, 128])
torch.Size([16, 196, 160])
torch.Size([16, 49, 320])
torch.Size([16, 49, 320])


In [15]:
import torch
import torch.nn as nn
        
class TinyViT_det(nn.Module):
    def __init__(self):
        super().__init__()
        from TinyViT.models.tiny_vit import tiny_vit_5m_224
        self.backbone = tiny_vit_5m_224(pretrained=True)
        
        self.up1 = nn.Upsample(scale_factor=2, mode='nearest')
        
    def forward(self, x):
        # Get backbone features (example shapes)
        features = self.backbone.forward_features(x)  # shape: (3, 197, 192) for TinyViT-5M

        b = features[0].shape[0]
        # Reshape to spatial format
        # Reshape to (batch, channels, height, width)
        shapes = [(28, 28), (14, 14), (7, 7), (7, 7)]
        spatial_features = [ feat.permute(0, 2, 1).reshape(b, -1, *shapes[i])
            for i, feat in enumerate(features)]
        #for i in spatial_features:
        #    print(i.shape)
        
        spatial_features[2] = self.up1(spatial_features[2])
        spatial_features[3] = self.up1(spatial_features[3])
        #print("after")
        #for i in spatial_features:
        #    print(i.shape)
            
        residual_output = spatial_features[0]
        x = torch.cat((spatial_features[1], spatial_features[2], spatial_features[3]), dim=1)
        #############################################################################################
        scale1 = self.conv_final1(x)
        #scale1_box = self.conv_final1_box(x)
        #scale1_cls = self.conv_final1_cls(x)
        #scale1 = torch.cat((scale1_box, scale1_cls), dim=1)
        
        output1, layer_loss = self.detection1(scale1, targets, self.gamma, self.alpha)
        loss += layer_loss
        
        x = self.upsample(x)
        x = torch.cat((x, residual_output), dim=1)
        
        scale2 = self.conv_final2(x)
        #scale2_box = self.conv_final2_box(x)
        #scale2_cls = self.conv_final2_cls(x)
        #scale2 = torch.cat((scale2_box, scale2_cls), dim=1)
        
        output2, layer_loss = self.detection2(scale2, targets, self.gamma, self.alpha)
        loss += layer_loss

        outputs = torch.cat([output1.detach(), output2.detach()], 1)
        return outputs if targets is None else (loss, outputs)

In [18]:
tiny = TinyViT_det()


In [19]:
out = tiny(imgs)
#out['cls'].shape

AttributeError: 'TinyViT_det' object has no attribute 'conv_final1'

In [29]:
out['reg'][0].shape

torch.Size([3, 36, 28, 28])

In [26]:
len(out['reg_preds'][0])

KeyError: 'reg_preds'

In [20]:
#from transformers import PyViTModel, PyViTConfig
from transformers import AutoImageProcessor, PvtForImageClassification
from transformers import AutoModelForImageClassification, AutoImageProcessor

pvt = AutoModelForImageClassification.from_pretrained("OpenGVLab/pvt_v2_b0")

In [21]:
class PVTFeatureExtractor(nn.Module):
    """
    Feature extractor using Pyramid Vision Transformer (PVT) from HuggingFace
    """
    def __init__(self, model_name: str = "OpenGVLab/pvt_v2_b0", pretrained: bool = True):
        super().__init__()
        # Load PVT model from HuggingFace
        self.pvt = AutoModelForImageClassification.from_pretrained("OpenGVLab/pvt_v2_b0")
        
        # Get feature dimensions from the model
        self.feature_dim = self.pvt.config.hidden_sizes
        self.num_stages = 4  # PVT typically has 4 feature map stages
        


In [22]:
imgs.shape

torch.Size([16, 3, 224, 224])

In [23]:
pvt = AutoModelForImageClassification.from_pretrained("OpenGVLab/pvt_v2_b0")

In [27]:
out = pvt(imgs)
out.logits.shape

torch.Size([16, 1000])

In [30]:
!git clone https://github.com/whai362/PVT.git

Cloning into 'PVT'...
remote: Enumerating objects: 1113, done.
remote: Counting objects: 100% (378/378), done.
remote: Compressing objects: 100% (113/113), done.
remote: Total 1113 (delta 339), reused 274 (delta 265), pack-reused 735 (from 1)
Receiving objects: 100% (1113/1113), 14.83 MiB | 23.73 MiB/s, done.
Resolving deltas: 100% (743/743), done.


In [79]:
import timm

print(timm.list_models('*pvt*'))

model = timm.create_model('pvt_v2_b0', pretrained=True)

['pvt_v2_b0', 'pvt_v2_b1', 'pvt_v2_b2', 'pvt_v2_b2_li', 'pvt_v2_b3', 'pvt_v2_b4', 'pvt_v2_b5', 'twins_pcpvt_base', 'twins_pcpvt_large', 'twins_pcpvt_small']


In [80]:
model = timm.create_model('pvt_v2_b0', pretrained=True, features_only=True)
features = model(x)  # Returns a list of features (if supported)

for i, feat in enumerate(features):
        print(f"Stage {i+1}: {feat.shape}")

Stage 1: torch.Size([1, 32, 56, 56])
Stage 2: torch.Size([1, 64, 28, 28])
Stage 3: torch.Size([1, 160, 14, 14])
Stage 4: torch.Size([1, 256, 7, 7])


In [8]:
#!pip install mmdet
#!pip install -U openmim
#!mim install mmcv-full
!pip uninstall mmcv mmcv-full -y


In [13]:
import torch
print(torch.__version__)

!pip install mmcv-full==2.1.0 -f https://download.openmmlab.com/mmcv/dist/cu121/torch2.3.0/index.html

2.3.0+cu121
Looking in links: https://download.openmmlab.com/mmcv/dist/cu121/torch2.3.0/index.html
ERROR: Could not find a version that satisfies the requirement mmcv-full==2.1.0 (from versions: 1.0rc0, 1.0rc2, 1.0.0, 1.0.1, 1.0.2, 1.0.3, 1.0.4, 1.0.5, 1.1.0, 1.1.1, 1.1.2, 1.1.3, 1.1.4, 1.1.5, 1.1.6, 1.2.0, 1.2.1, 1.2.2, 1.2.3, 1.2.4, 1.2.5, 1.2.6, 1.2.7, 1.3.0, 1.3.1, 1.3.3, 1.3.4, 1.3.5, 1.3.6, 1.3.7, 1.3.8, 1.3.9, 1.3.10, 1.3.11, 1.3.12, 1.3.13, 1.3.14, 1.3.15, 1.3.16, 1.3.17, 1.3.18, 1.4.0, 1.4.1, 1.4.2, 1.4.3, 1.4.4, 1.4.5, 1.4.6, 1.4.7, 1.4.8, 1.5.0, 1.5.1, 1.5.2, 1.5.3, 1.6.0, 1.6.1, 1.6.2, 1.7.0, 1.7.1, 1.7.2)
ERROR: No matching distribution found for mmcv-full==2.1.0


In [1]:
import shutil
root = '/home/a01328525/'
filename = root+"/transformer_object_dectection/TinyViT.zip"

# Target directory
extract_dir = root+"/transformer_object_dectection"

# Format of archive file
archive_format = "zip"

# Unpack the archive file 
shutil.unpack_archive(filename, extract_dir, archive_format) 
print("Archive file unpacked successfully.")

Archive file unpacked successfully.


In [34]:
import torch
import torch.nn as nn
import timm

class PVTBackbone(nn.Module):
    def __init__(self, model_name='pvt_v2_b0', pretrained=True):
        super().__init__()
        self.model = timm.create_model(model_name, pretrained=pretrained, features_only=True)

    def forward(self, x):
 
        features = self.model(x)
        
        return features

In [35]:
#imgs.shape
model = PVTBackbone()
model(imgs)[0].shape

torch.Size([16, 32, 56, 56])

In [37]:
class RetinaNetHead(nn.Module):
    def __init__(self, in_channels, num_anchors, num_classes):
        super().__init__()
        self.num_classes = num_classes
        self.num_anchors = num_anchors
        
        # Classification subnet
        self.cls_head = nn.Sequential(
            nn.Conv2d(in_channels, in_channels, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(in_channels, in_channels, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(in_channels, num_anchors * num_classes, 3, padding=1)
        )
        
        # Regression subnet
        self.reg_head = nn.Sequential(
            nn.Conv2d(in_channels, in_channels, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(in_channels, in_channels, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(in_channels, num_anchors * 4, 3, padding=1)
        )
        
        # Initialize weights
        for layer in [self.cls_head, self.reg_head]:
            for m in layer.modules():
                if isinstance(m, nn.Conv2d):
                    nn.init.normal_(m.weight, std=0.01)
                    nn.init.constant_(m.bias, 0)
        
        # Use prior for classification bias
        prior = 0.01
        self.cls_head[-1].bias.data.fill_(-torch.log(torch.tensor((1.0 - prior) / prior)))

    def forward(self, x):
        cls_output = self.cls_head(x)
        reg_output = self.reg_head(x)
        
        # Reshape outputs
        B, _, H, W = cls_output.shape
        cls_output = cls_output.view(B, -1, self.num_classes, H, W).permute(0, 3, 4, 1, 2)
        reg_output = reg_output.view(B, -1, 4, H, W).permute(0, 3, 4, 1, 2)
        
        return cls_output, reg_output

In [42]:
import torch
import math

class AnchorGenerator:
    def __init__(self, sizes=((32,), (64,), (128,), (256,)), aspect_ratios=((0.5, 1.0, 2.0),) * 4):
        # Convert sizes and ratios to tensors
        self.sizes = [torch.tensor(s) for s in sizes]
        self.aspect_ratios = [torch.tensor(r) for r in aspect_ratios]
    
    def generate_anchors(self, grid_sizes, strides):
        """Generate anchors for all feature maps.
        Args:
            grid_sizes: List of (H, W) tuples for each feature map
            strides: List of strides (e.g., [4, 8, 16, 32])
        Returns:
            Tensor of shape (N, 4) where N = sum(H*W*num_anchors)
        """
        anchors = []
        for (h, w), stride, size, ratios in zip(grid_sizes, strides, self.sizes, self.aspect_ratios):
            # Generate anchors for one feature map level
            cell_anchors = self._generate_cell_anchors(size, ratios)  # Shape: (A, 4)
            grid = self._create_grid(h, w, stride, cell_anchors)       # Shape: (H*W*A, 4)
            anchors.append(grid)
        return torch.cat(anchors, dim=0)  # Combined anchors for all levels
    
    def _generate_cell_anchors(self, size, ratios):
        """Generate anchor boxes for one grid cell."""
        scales = torch.tensor([1.0])  # Single scale per level
        anchors = []
        for ratio in ratios:
            w = size * torch.sqrt(ratio)
            h = size / torch.sqrt(ratio)
            # [xmin, ymin, xmax, ymax] format
            anchors.append(torch.stack([-w/2, -h/2, w/2, h/2]))
        return torch.stack(anchors) * scales.view(-1, 1, 1)  # Shape: (A, 4)
    
    def _create_grid(self, h, w, stride, cell_anchors):
        """Create grid of anchors for one feature map."""
        shifts_x = torch.arange(0, w) * stride
        shifts_y = torch.arange(0, h) * stride
        shift_y, shift_x = torch.meshgrid(shifts_y, shifts_x, indexing='ij')
        shifts = torch.stack([shift_x, shift_y, shift_x, shift_y], dim=-1)  # Shape: (H, W, 4)
        
        # Combine shifts with cell_anchors
        return (cell_anchors.view(1, 1, -1, 4) + shifts.view(h, w, 1, 4)).view(-1, 4)

# Usage Example
anchor_gen = AnchorGenerator()
grid_sizes = [(56, 56), (28, 28), (14, 14), (7, 7)]  # From PVT features
strides = [4, 8, 16, 32]  # Downsampling factors
anchors = anchor_gen.generate_anchors(grid_sizes, strides)  # Shape: [N, 4]
print(f"Generated {len(anchors)} anchors")

Generated 12495 anchors


In [48]:
class AnchorGenerator:
    def __init__(self, sizes=((32,), (64,), (128,), (256,)), aspect_ratios=((0.5, 1.0, 2.0),) * 4):
        self.sizes = [torch.tensor(s) for s in sizes]
        self.aspect_ratios = [torch.tensor(r) for r in aspect_ratios]
    
    def generate_anchors(self, grid_sizes, strides):
        anchors = []
        for (h, w), stride, size, ratios in zip(grid_sizes, strides, self.sizes, self.aspect_ratios):
            cell_anchors = self._generate_cell_anchors(size, ratios)
            grid = self._create_grid(h, w, stride, cell_anchors)
            anchors.append(grid)
        return torch.cat(anchors, dim=0)
    
    def _generate_cell_anchors(self, size, ratios):
        scales = torch.tensor([1.0])
        anchors = []
        for ratio in ratios:
            w = size * torch.sqrt(ratio)
            h = size / torch.sqrt(ratio)
            anchors.append(torch.stack([-w/2, -h/2, w/2, h/2]))
        return torch.stack(anchors) * scales.view(-1, 1, 1)
    
    def _create_grid(self, h, w, stride, cell_anchors):
        shifts_x = torch.arange(0, w) * stride
        shifts_y = torch.arange(0, h) * stride
        shift_y, shift_x = torch.meshgrid(shifts_y, shifts_x, indexing='ij')
        shifts = torch.stack([shift_x, shift_y, shift_x, shift_y], dim=-1)
        return (cell_anchors.view(1, 1, -1, 4) + shifts.view(h, w, 1, 4)).view(-1, 4)

In [49]:
class RetinaNetHead(nn.Module):
    def __init__(self, in_channels, num_anchors, num_classes):
        super().__init__()
        self.cls_head = nn.Sequential(
            nn.Conv2d(in_channels, in_channels, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(in_channels, num_anchors * num_classes, 3, padding=1)
        )
        self.reg_head = nn.Sequential(
            nn.Conv2d(in_channels, in_channels, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(in_channels, num_anchors * 4, 3, padding=1)
        )
        
        # Initialize weights
        for layer in [self.cls_head, self.reg_head]:
            nn.init.normal_(layer[-1].weight, std=0.01)
            nn.init.constant_(layer[-1].bias, 0)

    def forward(self, x):
        return self.cls_head(x), self.reg_head(x)

In [60]:
class RetinaNet(nn.Module):
    def __init__(self, num_classes=80, num_anchors=9):
        super().__init__()
        self.backbone = PVTBackbone()
        self.anchor_gen = AnchorGenerator()
        self.num_anchors = num_anchors
        
        # FPN (1x1 convs to unify channels)
        self.fpn = nn.ModuleList([
            nn.Conv2d(32, 256, 1),
            nn.Conv2d(64, 256, 1),
            nn.Conv2d(160, 256, 1),
            nn.Conv2d(256, 256, 1)
        ])
        
        # Heads
        self.heads = nn.ModuleList([
            RetinaNetHead(256, num_anchors, num_classes) for _ in range(4)
        ])

    def forward(self, x):
        # Backbone features
        features = self.backbone(x)
        
        # FPN
        fpn_features = [conv(feat) for feat, conv in zip(features, self.fpn)]
        
        
        # Generate anchors
        grid_sizes = [f.shape[-2:] for f in fpn_features]
        strides = [4, 8, 16, 32]  # For 224x224 input
        anchors = self.anchor_gen.generate_anchors(grid_sizes, strides)
        
        # Heads
        cls_preds, reg_preds = [], []
        for feat, head in zip(fpn_features, self.heads):
            cls, reg = head(feat)
            cls_preds.append(cls)
            reg_preds.append(reg)
        
        return {
            'cls': cls_preds,      # List of [B, A*C, H, W]
            'reg': reg_preds,      # List of [B, A*4, H, W]
            'anchors': anchors     # [N, 4]
        }

In [63]:
model = RetinaNet(num_classes=80)
out = model(imgs)
reg_preds = out['reg']
anchors = out['anchors']

In [69]:
anchors

tensor([[-11.3137, -22.6274,  11.3137,  22.6274],
        [-16.0000, -16.0000,  16.0000,  16.0000],
        [-22.6274, -11.3137,  22.6274,  11.3137],
        ...,
        [101.4903,  10.9807, 282.5097, 373.0193],
        [ 64.0000,  64.0000, 320.0000, 320.0000],
        [ 10.9807, 101.4903, 373.0193, 282.5097]])

In [70]:
def decode_boxes(anchors, reg_preds):
    """Convert reg outputs to final box coordinates."""
    # Anchors: [N, 4] (x_center, y_center, w, h)
    # reg_preds: [N, 4] (dx, dy, dw, dh)
    boxes = torch.zeros_like(reg_preds)
    boxes[:, 0] = anchors[:, 0] + reg_preds[:, 0] * anchors[:, 2]  # x_center
    boxes[:, 1] = anchors[:, 1] + reg_preds[:, 1] * anchors[:, 3]  # y_center
    boxes[:, 2] = anchors[:, 2] * torch.exp(reg_preds[:, 2])       # width
    boxes[:, 3] = anchors[:, 3] * torch.exp(reg_preds[:, 3])       # height
    return boxes  # [x_center, y_center, w, h]

def decode_all_boxes(anchors, reg_preds_list):
    """Decode predictions for ALL FPN levels."""
    decoded_boxes = []
    for reg_preds in reg_preds_list:
        # Reshape reg_preds from (B, A*4, H, W) to (B, H, W, A, 4)
        B, _, H, W = reg_preds.shape
        reg_preds = reg_preds.view(B, -1, 4, H, W).permute(0, 3, 4, 1, 2)
        
        # Decode boxes for this level
        level_boxes = decode_boxes(anchors, reg_preds.reshape(-1, 4))
        decoded_boxes.append(level_boxes)
    
    return torch.cat(decoded_boxes, dim=0)  # Combine all levels

In [71]:
decode_all_boxes(anchors, reg_preds)

RuntimeError: The size of tensor a (451584) must match the size of tensor b (12495) at non-singleton dimension 0

In [53]:
from torchvision.ops import sigmoid_focal_loss

class RetinaNetLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        
    def forward(self, preds, targets):
        cls_preds = torch.cat([p.flatten(start_dim=2) for p in preds['cls']], dim=2)
        reg_preds = torch.cat([p.flatten(start_dim=2) for p in preds['reg']], dim=2)
        anchors = torch.cat([a.flatten(start_dim=1) for a in preds['anchors']], dim=1)
        
        # Match anchors to targets
        matched_targets = self.match_anchors(anchors, targets)
        
        # Classification loss (Focal Loss)
        cls_loss = sigmoid_focal_loss(
            cls_preds,
            matched_targets['labels'],
            alpha=self.alpha,
            gamma=self.gamma,
            reduction='sum'
        )
        
        # Regression loss (Smooth L1)
        pos_mask = matched_targets['labels'] > 0
        reg_loss = nn.functional.smooth_l1_loss(
            reg_preds[pos_mask],
            matched_targets['boxes'][pos_mask],
            reduction='sum'
        )
        
        # Normalize by number of positive anchors
        num_pos = max(1, pos_mask.sum().item())
        return {'cls_loss': cls_loss / num_pos, 'reg_loss': reg_loss / num_pos}
    
    def match_anchors(self, anchors, targets):
        # Implement anchor matching (simplified)
        # In practice, use torchvision.ops.box_iou and assign_targets_to_anchors
        return {
            'labels': torch.zeros_like(anchors[:, :, 0]),  # Placeholder
            'boxes': torch.zeros_like(anchors)             # Placeholder
        }


In [59]:
preds['anchors'].shape

torch.Size([12495, 4])

In [56]:
model = RetinaNet(num_classes=1)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
criterion = RetinaNetLoss()

for P, images, targets,_ in train_dataloader:  # targets: list of dicts {'boxes': ..., 'labels': ...}
    preds = model(images)
    print(preds.shape)
    ptint(targets.shape)
    loss_dict = criterion(preds, targets)
    loss = loss_dict['cls_loss'] + loss_dict['reg_loss']
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

AttributeError: 'dict' object has no attribute 'shape'